# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 90b2ed1c-a6ee-40b5-a278-c64c58ce4354
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 90b2ed1c-a6ee-40b5-a278-c64c58ce4354 to get into ready status...
Session 90b2ed1c-a6ee-40b5-a278-c64c58ce4354 

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)

In [40]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='freelancing', table_name='raw_data')
dyf.printSchema()

root
|-- freelancer_id: string
|-- name: string
|-- gender: string
|-- age: double
|-- country: string
|-- language: string
|-- primary_skill: string
|-- years_of_experience: double
|-- hourly_rate (usd): string
|-- is_active: string
|-- client_satisfaction: string
|-- rating: double


In [51]:
df = dyf.toDF()
df.show(3)

+-------------+---------------+------+----+---------+--------+--------------------+-------------------+-----------------+---------+-------------------+------+
|freelancer_id|           name|gender| age|  country|language|       primary_skill|years_of_experience|hourly_rate (usd)|is_active|client_satisfaction|rating|
+-------------+---------------+------+----+---------+--------+--------------------+-------------------+-----------------+---------+-------------------+------+
|     FL250001|Ms. Nicole Kidd|     f|52.0|    Italy| Italian|Blockchain Develo...|               11.0|              100|        0|                   |  NULL|
|     FL250002| Vanessa Garcia|FEMALE|52.0|Australia| English|         Mobile Apps|               34.0|          USD 100|        1|                84%|   3.3|
|     FL250003|    Juan Nelson|  male|53.0|  Germany|  German|      Graphic Design|               31.0|               50|        N|                71%|   0.0|
+-------------+---------------+------+----+---

In [52]:
df.dtypes

[('freelancer_id', 'string'), ('name', 'string'), ('gender', 'string'), ('age', 'double'), ('country', 'string'), ('language', 'string'), ('primary_skill', 'string'), ('years_of_experience', 'double'), ('hourly_rate (usd)', 'string'), ('is_active', 'string'), ('client_satisfaction', 'string'), ('rating', 'double')]


In [53]:
from pyspark.sql.functions import when, col, upper,initcap,lower,regexp_replace,trim
df = df.withColumn(
    "gender",
    when(upper(col("gender")) == "F", "Female")
    .when(upper(col("gender")) == "M", "Male")
    .otherwise(col("gender"))
)

In [54]:
df = df.withColumn(
    "gender",
    initcap(lower(col("gender")))
    )

In [55]:
df = df.withColumn(
    "age",
    col("age").cast("int")
    ).withColumn(
    "years_of_experience",
    col("years_of_experience").cast("int")
    ).withColumn(
    "hourly_rate (usd)",
    regexp_replace(col("hourly_rate (usd)"), "[^0-9]", "")
    ).withColumn(
    "hourly_rate (usd)",
    col("hourly_rate (usd)").cast("int")
    )

In [56]:
df = df.fillna({"hourly_rate (usd)" : 0, "years_of_experience" : 0})

In [57]:
df.show()

+-------------+---------------+------+---+--------------+----------+--------------------+-------------------+-----------------+---------+-------------------+------+
|freelancer_id|           name|gender|age|       country|  language|       primary_skill|years_of_experience|hourly_rate (usd)|is_active|client_satisfaction|rating|
+-------------+---------------+------+---+--------------+----------+--------------------+-------------------+-----------------+---------+-------------------+------+
|     FL250001|Ms. Nicole Kidd|Female| 52|         Italy|   Italian|Blockchain Develo...|                 11|              100|        0|                   |  NULL|
|     FL250002| Vanessa Garcia|Female| 52|     Australia|   English|         Mobile Apps|                 34|              100|        1|                84%|   3.3|
|     FL250003|    Juan Nelson|  Male| 53|       Germany|    German|      Graphic Design|                 31|               50|        N|                71%|   0.0|
|     FL25

In [58]:
df.select("is_active").distinct().show()

+---------+
|is_active|
+---------+
|        N|
|        Y|
|        0|
|     True|
|      yes|
|    False|
|        1|
|       no|
|         |
+---------+


In [61]:
df = df.withColumn(
    "is_active",
    when(lower(trim(col("is_active"))).isin("yes", "true","y", "1"), "Yes")
    .when(lower(trim(col("is_active"))).isin("no", "false","n", "0"), "No")
)

In [62]:
df.select("is_active").distinct().show()

+---------+
|is_active|
+---------+
|     NULL|
|      Yes|
|       No|
+---------+


In [63]:
df.groupBy("is_active").count().show()

+---------+-----+
|is_active|count|
+---------+-----+
|     NULL|   89|
|      Yes|  446|
|       No|  465|
+---------+-----+


In [67]:
df = df.withColumn(
    "client_satisfaction",
    regexp_replace(col("client_satisfaction"), "[^0-9]", "")
    ).withColumn(
    "client_satisfaction",
    col("client_satisfaction").cast("int")
    )

In [68]:
df = df.fillna(
    {"client_satisfaction" : "0"}
    )

In [69]:
df.show()

+-------------+---------------+------+---+--------------+----------+--------------------+-------------------+-----------------+---------+-------------------+------+
|freelancer_id|           name|gender|age|       country|  language|       primary_skill|years_of_experience|hourly_rate (usd)|is_active|client_satisfaction|rating|
+-------------+---------------+------+---+--------------+----------+--------------------+-------------------+-----------------+---------+-------------------+------+
|     FL250001|Ms. Nicole Kidd|Female| 52|         Italy|   Italian|Blockchain Develo...|                 11|              100|       No|                  0|  NULL|
|     FL250002| Vanessa Garcia|Female| 52|     Australia|   English|         Mobile Apps|                 34|              100|      Yes|                 84|   3.3|
|     FL250003|    Juan Nelson|  Male| 53|       Germany|    German|      Graphic Design|                 31|               50|       No|                 71|   0.0|
|     FL25

In [70]:
df = df.withColumnRenamed("client_satisfaction", "client_satisfaction (%)")

In [71]:
df.printSchema()

root
 |-- freelancer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- language: string (nullable = true)
 |-- primary_skill: string (nullable = true)
 |-- years_of_experience: integer (nullable = false)
 |-- hourly_rate (usd): integer (nullable = false)
 |-- is_active: string (nullable = true)
 |-- client_satisfaction (%): integer (nullable = true)
 |-- rating: double (nullable = true)


In [73]:
df.write.mode("overwrite").parquet("s3://myfirst-bucket-126606499301-eu-north-1-an/Processed-Data/")